In [1]:
"""Smoke-test for the NEW Rossmann forecasting pipeline (hypertuning).

Runs the full feature-engineering -> target-transform -> inner-CV objective and
refit flow on a few real stores, then exercises hypertuning.optimize end-to-end
to validate the updated TimeSeriesCV(n_splits, train_size, test_size) API.
"""
import os
import logging
import optuna
import pandas as pd

from src import features
from src.preprocessing import preprocess_data
from src.engine.target_transformer import TargetTransformer
from src.settings import AppSettings

config = AppSettings.from_yaml('./config.yaml')

os.makedirs(config.path.log_dir, exist_ok=True)
assert os.path.exists(config.path.data_dir), \
    f"Data directory {config.path.data_dir} does not exist."

optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.basicConfig(level=logging.INFO,
                    filename=config.path.logs,
                    format="%(levelname)s  %(message)s")

logger = logging.getLogger(__name__)




c:\Users\m_kal\Downloads\rossmann_store_sales\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sales = pd.read_csv(config.path.train)
stores = pd.read_csv(config.path.stores)

stores_to_use = [1, 2, 3]
stores = stores[stores['Store'].isin(stores_to_use)]

df = preprocess_data(sales, stores)

C:\Users\m_kal\AppData\Local\Temp\ipykernel_4636\288647734.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  sales = pd.read_csv(config.path.train)


In [39]:
y = df.set_index(['Store', 'Date'])['Sales']

X = (df
     .set_index(['Date'])
     .groupby('Store')
     .apply(lambda df: features.compute(df, config.feature_engineering, config.horizon)))

trf = TargetTransformer(forecast_horizon=pd.DateOffset(days=-config.horizon.days),
                        anchor_col='lag_days_0')
trf.fit(X)

y = trf.transform(y)    # forward difference target
X = X.loc[y.index]      # align features with target

C:\Users\m_kal\AppData\Local\Temp\ipykernel_4636\1219067114.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: features.compute(df, config.feature_engineering, config.horizon)))


In [173]:
df_ = df.set_index(['Date']).groupby('Store')['isStateHoliday']

for _, group in df_:
    offset = config.horizon

In [174]:
from src.features.store import _dates_to_days, _days_to_holiday, _days_since_holiday
import numpy as np

group_sorted = group.sort_index()


is_holiday = group_sorted == True
holiday_days = _dates_to_days(group_sorted.index[is_holiday])
lookup_days  = _dates_to_days(group_sorted.index + offset)  # shift only the lookup dates

if len(holiday_days) > 0:
    days_to_next    = _days_to_holiday(holiday_days, lookup_days)
    days_since_last = _days_since_holiday(holiday_days, lookup_days)
else:
    days_to_next = np.full(len(group), np.nan)  # Array filled with NaN values
    days_since_last = np.full(len(group), np.nan)

res = pd.DataFrame(data=np.stack([days_to_next, days_since_last], axis=-1),
                    index=group_sorted.index,
                    columns=['DaysToNextHoliday', 'DaysSinceLastHoliday'])

res.reindex(group.index)

,DaysToNextHoliday,DaysSinceLastHoliday
Date,,
2013-01-01,77.0,10.0
2013-01-02,76.0,11.0
2013-01-03,75.0,12.0
2013-01-04,74.0,13.0
2013-01-05,73.0,14.0
...,...,...
2015-07-27,NaN,63.0
2015-07-28,NaN,64.0
2015-07-29,NaN,65.0


In [175]:
df_ = df.set_index(['Date']).groupby('Store')['isStateHoliday']

for _, group in df_:
    offset = config.horizon


date_array = group.index.to_numpy()

# Mark dates that are holidays; NaT elsewhere
holiday_dates = pd.Series(
    data=np.where(group == 1, date_array, pd.NaT),
    index=group.index,
    dtype='datetime64[ns]'
)

# Propagate last/next holiday dates to every row
last_holiday = holiday_dates.ffill()
next_holiday = holiday_dates.bfill()

# Compute signed day intervals
dates = group.index.to_series()
days_since = (dates - last_holiday).dt.days.fillna(999).astype(int)
days_until = (next_holiday - dates).dt.days.fillna(999).astype(int)

holiday_dist = pd.DataFrame({'DaysSinceHoliday': days_since, 'DaysToNextHoliday': days_until})


In [176]:
# Smooth distribution wave formulation
sigma = 3 # Defines a multi-day decaying envelope of impact
holiday_dist['Pre_Holiday_Wave'] = np.exp(-(holiday_dist['DaysToNextHoliday']**2) / (2 * sigma**2))
holiday_dist['Post_Holiday_Wave'] = np.exp(-(holiday_dist['DaysSinceHoliday']**2) / (2 * sigma**2))

eps = np.sqrt(np.finfo(float).eps)
wave_cols = ['Pre_Holiday_Wave', 'Post_Holiday_Wave']
holiday_dist[wave_cols] = holiday_dist[wave_cols].where(holiday_dist[wave_cols] >= eps, 0.0)


In [177]:
# Offset the lookup dates by the specified offset to align with the forecast horizon, 
# i.e. if the forecast horizon is 7 days, we want to know how many days to the next holiday are 
# in 7 days from now - at the forecast date.
lookup_index = group.index + offset


In [178]:
holiday_dist.head(5)

,DaysSinceHoliday,DaysToNextHoliday,Pre_Holiday_Wave,Post_Holiday_Wave
Date,,,,
2013-01-01,0,0,1.0,1.000000
2013-01-02,1,86,0.0,0.945959
2013-01-03,2,85,0.0,0.800737
2013-01-04,3,84,0.0,0.606531
2013-01-05,4,83,0.0,0.411112


In [179]:
holiday_dist = holiday_dist.reindex(lookup_index)
holiday_dist.index = group.index

In [180]:
holiday_dist

,DaysSinceHoliday,DaysToNextHoliday,Pre_Holiday_Wave,Post_Holiday_Wave
Date,,,,
2013-01-01,10.0,77.0,0.0,0.003866
2013-01-02,11.0,76.0,0.0,0.001204
2013-01-03,12.0,75.0,0.0,0.000335
2013-01-04,13.0,74.0,0.0,0.000084
2013-01-05,14.0,73.0,0.0,0.000019
...,...,...,...,...
2015-07-27,NaN,NaN,NaN,NaN
2015-07-28,NaN,NaN,NaN,NaN
2015-07-29,NaN,NaN,NaN,NaN
